# Notebook 07 : Séquences d'erreurs
# Error Sequences

**Projet :** OJS Submission Analytics, Ritsumeikan University, DDSE Laboratory
**Auteur / Author :** Raphael BELY
**Données / Data :** AtCoder ABC (Project CodeNet, IBM 2021), `src/main.py` PHASE 7

---

Ce notebook analyse les **chemins complets de soumissions** (ex. TLE → WA → AC) pour chaque paire
(utilisateur, problème) dont la première soumission n'était pas AC.

This notebook analyses the **full submission paths** (e.g. TLE → WA → AC) for every (user, problem)
pair whose first submission was not AC.

**CSV utilisé :** `data/processed/atcoder_error_sequences.csv`

**Structure :**
- **S0** : Vue d'ensemble — Top 10 séquences par niveau de difficulté, tous groupes confondus / Top 10 error sequences by difficulty level
- **S1** : Taux de résolution par combinaison d'erreurs × groupe (niveau D) — extension originale de Shimizu : stratification par compétence / Resolution rate by error combination × group (Shimizu-style figure)
- **S1b** : Version interactive Plotly de S1 avec survol / Interactive Plotly version of S1
- **S2** : Diagrammes de Sankey — flux depuis chaque première erreur (WA, TLE, CE, RE), niveau D / Sankey diagrams by first error type, level D
- **S3** : Diagrammes de Sankey — flux depuis TLE comparé entre groupes G4/G5/G6, niveau D / Sankey diagrams by group for TLE, level D
- **S4** : Répartition des chemins complets depuis TLE × groupe — bar charts comparatifs avec pourcentages / Complete path distribution from TLE × group
- **S5** : Taux de résolution selon la longueur du chemin (nombre de transitions depuis TLE) × groupe / Resolution rate by path length × group


In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR    = Path("../data/processed/")
FIGURES_DIR = Path("../figures/")
FIGURES_DIR.mkdir(exist_ok=True)

# Palette cohérente avec les autres notebooks / Consistent color palette
ERR_COLORS = {
    "WA":      "#e74c3c",
    "TLE":     "#f39c12",
    "CE":      "#3498db",
    "RE":      "#9b59b6",
    "AC":      "#2ecc71",
    "Abandon": "#95a5a6",
    "Other":   "#7f8c8d",
}

DIFFICULTIES = ["A", "B", "C", "D", "E", "F"]
FIRST_ERRORS = ["WA", "TLE", "CE", "RE"]
GROUPS       = ["G4", "G5", "G6"]

In [ ]:
df_seq = pl.read_csv(DATA_DIR / "atcoder_error_sequences.csv")

print(f"Shape    : {df_seq.shape[0]:,} lignes × {df_seq.shape[1]} colonnes")
print(f"Colonnes : {df_seq.columns}")
print(f"Niveaux  : {sorted(df_seq['difficulty'].unique().to_list())}")
print(f"Groupes  : {sorted(df_seq['proficiency_group'].drop_nulls().unique().to_list())}")
print(f"\nTotal paires (user, problème) : {df_seq['n_pairs'].sum():,}")
df_seq.head(8)

---
## Section 0 — Vue d'ensemble : Top séquences par niveau de difficulté
## Overview: Top Sequences by Difficulty Level


In [ ]:
TOP_N = 10

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, diff in enumerate(DIFFICULTIES):
    ax = axes[i]

    # Agrège sur tous les groupes et toutes les premières erreurs
    top_df = (
        df_seq
        .filter(pl.col("difficulty") == diff)
        .group_by(["sequence", "resolved"])
        .agg(pl.col("n_pairs").sum())
        .sort("n_pairs", descending=True)
        .head(TOP_N)
    )

    if top_df.is_empty():
        ax.set_visible(False)
        continue

    seqs   = top_df["sequence"].to_list()
    counts = top_df["n_pairs"].to_list()
    res    = top_df["resolved"].to_list()
    colors = ["#2ecc71" if r else "#95a5a6" for r in res]

    bars = ax.barh(range(len(seqs)), counts, color=colors,
                   edgecolor="white", linewidth=0.5, height=0.7)
    ax.set_yticks(range(len(seqs)))
    ax.set_yticklabels(seqs, fontsize=8.5, fontfamily="monospace")
    ax.invert_yaxis()
    ax.set_xlabel("Paires (user, problème) / (user, problem) pairs", fontsize=8)
    ax.set_title(f"Niveau / Level {diff}", fontweight="bold", fontsize=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.spines[["top", "right"]].set_visible(False)

    max_val = max(counts)
    for bar, n in zip(bars, counts):
        ax.text(
            bar.get_width() + max_val * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{n:,}", va="center", ha="left", fontsize=7.5,
        )

legend_elements = [
    mpatches.Patch(facecolor="#2ecc71", label="Résolu / Resolved (→ AC)"),
    mpatches.Patch(facecolor="#95a5a6", label="Abandonné / Abandoned"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.03))

fig.suptitle(
    "Top 10 des séquences de soumissions les plus fréquentes par niveau de difficulté\n"
    "Top 10 most frequent submission sequences by difficulty level",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_s1_top_sequences.png", dpi=150, bbox_inches="tight")
plt.show()

*Analyse à compléter après exécution / Analysis to be filled after running.*

---
## Section 1 — Résolution par combinaison d'erreurs × groupe, niveau D
## Resolution Rate by Error Combination × Group, Level D


In [ ]:
import numpy as np

TERMINAL    = {"AC"}
ERROR_ORDER = {"WA": 0, "CE": 1, "RE": 2, "TLE": 3, "Other": 4}
MIN_SAMPLE  = 20

def get_error_combination(seq):
    errors = set(seq.split("→")) - TERMINAL
    errors -= {"Other"}
    if not errors:
        return None
    return ", ".join(sorted(errors, key=lambda x: ERROR_ORDER.get(x, 99)))

def plot_resolution_by_combo(difficulty, groups, group_colors, min_total=50):
    df_filt = df_seq.filter(pl.col("difficulty") == difficulty)
    df_filt = df_filt.with_columns(
        pl.Series("error_combination",
                  [get_error_combination(s) for s in df_filt["sequence"].to_list()])
    ).filter(pl.col("error_combination").is_not_null())

    df_agg = (
        df_filt
        .group_by(["error_combination", "proficiency_group"])
        .agg([
            pl.col("n_pairs").sum().alias("total"),
            pl.when(pl.col("resolved")).then(pl.col("n_pairs")).otherwise(0)
              .sum().alias("resolved_n"),
        ])
        .with_columns(
            (pl.col("resolved_n") / pl.col("total") * 100).round(1).alias("resolution_pct")
        )
    )

    combo_order = (
        df_filt.group_by("error_combination")
        .agg(pl.col("n_pairs").sum().alias("total"))
        .filter(pl.col("total") >= min_total)
        .sort("total", descending=True)
        ["error_combination"].to_list()
    )

    # Dict de lookup O(1)
    lookup = {
        (r["error_combination"], r["proficiency_group"]): (r["resolution_pct"], r["total"])
        for r in df_agg.filter(pl.col("proficiency_group").is_in(groups)).iter_rows(named=True)
    }

    n_combos = len(combo_order)
    n_groups = len(groups)
    bar_h    = min(0.20, 0.8 / n_groups)
    y_pos    = np.arange(n_combos)

    fig, ax = plt.subplots(figsize=(10, max(5, n_combos * 0.65)))

    for gi, (group, color) in enumerate(zip(groups, group_colors)):
        offset = (gi - n_groups / 2 + 0.5) * bar_h
        ys, rs, stars = [], [], []

        for ci, combo in enumerate(combo_order):
            val = lookup.get((combo, group))
            if val is not None:
                rate, total = val
                ys.append(y_pos[ci] + offset)
                rs.append(rate)
                stars.append(total <= MIN_SAMPLE)

        if ys:
            ax.barh(ys, rs, height=bar_h * 0.9, color=color, label=group, zorder=2)
            for yp, rp, star in zip(ys, rs, stars):
                if star:
                    ax.text(rp + 0.8, yp, "*", color="red",
                            va="center", ha="left", fontsize=9, zorder=3)
        else:
            ax.barh([], [], color=color, label=group)

    for xv in [60, 80, 100]:
        ax.axvline(xv, color="#dddddd", linewidth=0.8, zorder=1)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(combo_order, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Resolution rate (%)", fontsize=10)
    ax.set_xlim(0, 108)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(title="Group", loc="lower right", ncol=3, fontsize=9)
    ax.set_title(
        f"Taux de résolution par combinaison d'erreurs et groupe, niveau {difficulty}\n"
        f"Resolution rate by error combination and proficiency group, level {difficulty}\n"
        f"(* effectif \u2264 20 / sample size \u2264 20)",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()


# ── Niveau A : tous groupes G1-G6 (reproduction Shimizu Figure 5) ─────────────
COLORS_6 = ["#d9d9d9", "#b3b3b3", "#808080", "#4d7fa0", "#2c5f7a", "#1a3a4d"]
plot_resolution_by_combo("A", ["G1", "G2", "G3", "G4", "G5", "G6"], COLORS_6)

# ── Niveau D : G4-G6 uniquement (G1-G3 ont 0 % résolution TLE, données insuffisantes) ──
COLORS_3 = ["#4d7fa0", "#2c5f7a", "#1a3a4d"]
plot_resolution_by_combo("D", ["G4", "G5", "G6"], COLORS_3)

*Analyse à compléter après exécution / Analysis to be filled after running.*

---
### S0b — Version interactive avec survol / Interactive version with hover (Plotly)

Même figure que S0 mais en Plotly : en passant la souris sur une barre,
on voit le **pourcentage exact** et le **nombre de paires** correspondant.

In [ ]:
import plotly.graph_objects as go

COLORS_6_HEX = ["#d9d9d9", "#b3b3b3", "#808080", "#4d7fa0", "#2c5f7a", "#1a3a4d"]
COLORS_3_HEX = ["#4d7fa0", "#2c5f7a", "#1a3a4d"]


def plot_resolution_interactive(difficulty, groups, group_colors, min_total=50):
    df_filt = df_seq.filter(pl.col("difficulty") == difficulty)
    df_filt = df_filt.with_columns(
        pl.Series("error_combination",
                  [get_error_combination(s) for s in df_filt["sequence"].to_list()])
    )
    df_agg = (
        df_filt
        .group_by(["error_combination", "proficiency_group"])
        .agg([
            pl.col("n_pairs").sum().alias("total"),
            pl.when(pl.col("resolved")).then(pl.col("n_pairs")).otherwise(0)
              .sum().alias("resolved_n"),
        ])
        .with_columns(
            (pl.col("resolved_n") / pl.col("total") * 100).round(1).alias("resolution_pct")
        )
    )
    combo_order = (
        df_filt.group_by("error_combination")
        .agg(pl.col("n_pairs").sum().alias("total"))
        .filter(pl.col("total") >= min_total)
        .sort("total", descending=True)
        ["error_combination"].to_list()
    )
    lookup = {
        (r["error_combination"], r["proficiency_group"]): (r["resolution_pct"], r["total"])
        for r in df_agg.filter(pl.col("proficiency_group").is_in(groups)).iter_rows(named=True)
    }

    fig = go.Figure()
    for group, color in zip(groups, group_colors):
        rates, totals = [], []
        for combo in combo_order:
            val = lookup.get((combo, group))
            rates.append(val[0] if val else None)
            totals.append(val[1] if val else 0)

        fig.add_trace(go.Bar(
            name=group,
            x=combo_order,
            y=rates,
            customdata=totals,
            marker_color=color,
            hovertemplate=(
                "<b>%{x}</b><br>"
                f"Groupe {group}<br>"
                "Résolution : <b>%{y:.1f}%</b><br>"
                "N = %{customdata:,}<extra></extra>"
            ),
        ))

    fig.update_layout(
        barmode="group",
        title=f"Taux de résolution par combinaison d'erreurs — Niveau {difficulty}",
        xaxis_title="Combinaison d'erreurs",
        yaxis_title="Taux de résolution (%)",
        yaxis_range=[0, 108],
        legend_title="Groupe",
        height=480,
        template="plotly_white",
    )
    fig.show()


# Niveau A — tous groupes
plot_resolution_interactive("A", ["G1", "G2", "G3", "G4", "G5", "G6"], COLORS_6_HEX)

# Niveau D — groupes experts uniquement
plot_resolution_interactive("D", ["G4", "G5", "G6"], COLORS_3_HEX)

---
## Section 2 : Diagrammes de Sankey, Niveau D × Première erreur
## Sankey Diagrams — Level D × First Error

**🇫🇷** Pour le niveau D (charnière entre débutant et expert dans notre corpus),
on visualise le flux complet de soumissions à partir de chaque type de première erreur (WA, TLE, CE, RE).
Chaque nœud représente une erreur à une position donnée dans la séquence
(WA à l'étape 1 et WA à l'étape 2 sont des nœuds distincts pour éviter les cycles).
Les séquences avec moins de `MIN_PAIRS_D` paires sont filtrées pour ne conserver que les chemins significatifs.

**🇬🇧** For level D (the transition point between novice and expert in our corpus),
we visualise the full submission flow starting from each first error type (WA, TLE, CE, RE).
Each node represents an error at a given step position
(WA at step 1 and WA at step 2 are distinct nodes to prevent cycles).
Sequences with fewer than `MIN_PAIRS_D` pairs are filtered to retain only significant paths.

In [ ]:
def hex_to_rgba(hex_color, alpha=0.9):
    """Convert 6-char hex color to plotly rgba() string."""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def build_sankey(
    df_seq,
    difficulty,
    first_error_filter=None,
    group_filter=None,
    min_pairs=30,
    max_depth=6,
    title="",
):
    """
    Build a Plotly Sankey diagram from aggregated error sequences.

    Each (error_type, step_position) is a distinct node to prevent cycles.
    'AC' and 'Abandon' are shared terminal nodes (position-independent).
    Sequences with n_pairs < min_pairs are filtered (noise reduction).

    Parameters
    ----------
    df_seq             : pl.DataFrame  output of compute_error_sequences()
    difficulty         : str           difficulty level filter ('A' to 'F')
    first_error_filter : str or None   restrict to one first error type
    group_filter       : str or None   restrict to one proficiency group
    min_pairs          : int           minimum pairs per sequence to display
    max_depth          : int           max sequence steps to show
    title              : str           figure title
    """
    mask = pl.col("difficulty") == difficulty
    if first_error_filter:
        mask = mask & (pl.col("first_error") == first_error_filter)
    if group_filter:
        mask = mask & (pl.col("proficiency_group") == group_filter)

    df = df_seq.filter(mask).filter(pl.col("n_pairs") >= min_pairs)
    if df.is_empty():
        return None

    # Construit les transitions (src_key, tgt_key) -> poids
    transitions = {}

    for row in df.iter_rows(named=True):
        steps = row["sequence"].split("→")  # split on →
        if not row["resolved"]:
            steps = steps + ["Abandon"]
        steps = steps[: max_depth + 1]
        n = row["n_pairs"]

        for pos in range(len(steps) - 1):
            src, tgt = steps[pos], steps[pos + 1]
            src_key = src if src in ("AC", "Abandon") else f"{src}_p{pos + 1}"
            tgt_key = tgt if tgt in ("AC", "Abandon") else f"{tgt}_p{pos + 2}"
            transitions[(src_key, tgt_key)] = transitions.get((src_key, tgt_key), 0) + n

    if not transitions:
        return None

    # Liste de noeuds ordonnée (ordre d'apparition)
    node_keys = list(dict.fromkeys(k for pair in transitions for k in pair))
    node_idx  = {k: i for i, k in enumerate(node_keys)}

    def base(key):
        return key.split("_p")[0]

    node_labels = [base(k) for k in node_keys]
    node_colors = [hex_to_rgba(ERR_COLORS.get(base(k), "#bdc3c7"), 0.85) for k in node_keys]
    link_colors = [hex_to_rgba(ERR_COLORS.get(base(s), "#bdc3c7"), 0.35) for s, _ in transitions]

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=12,
            thickness=18,
            line=dict(color="white", width=0.5),
            label=node_labels,
            color=node_colors,
        ),
        link=dict(
            source=[node_idx[s] for s, _ in transitions],
            target=[node_idx[t] for _, t in transitions],
            value=list(transitions.values()),
            color=link_colors,
        ),
    ))

    fig.update_layout(
        title_text=title,
        title_font_size=13,
        font_size=11,
        height=480,
        margin=dict(l=10, r=10, t=60, b=10),
    )
    return fig

In [ ]:
MIN_PAIRS_D = 50  # seuil bruit pour niveau D / noise threshold for level D

for fe in FIRST_ERRORS:
    total_pairs = (
        df_seq
        .filter((pl.col("difficulty") == "D") & (pl.col("first_error") == fe))
        ["n_pairs"].sum()
    )

    fig = build_sankey(
        df_seq,
        difficulty="D",
        first_error_filter=fe,
        min_pairs=MIN_PAIRS_D,
        title=(
            f"Niveau D, premiere erreur : {fe}  ({total_pairs:,} paires au total)\n"
            f"Level D, first error: {fe}  (sequences with >= {MIN_PAIRS_D} pairs shown)"
        ),
    )

    if fig is not None:
        fig.show()
    else:
        print(f"[{fe}] Aucune sequence avec n_pairs >= {MIN_PAIRS_D} au niveau D.")

*Analyse à compléter après exécution / Analysis to be filled after running.*

---
## Section 3 : Diagrammes de Sankey, Niveau D × Groupe (première erreur TLE)
## Sankey Diagrams — Level D × Proficiency Group (first error TLE)

**🇫🇷** On isole les paires dont la **première erreur est TLE** au niveau D
et on compare les chemins suivis par les groupes G4, G5 et G6.
Hypothèse issue du notebook 06 S4 : G4 erre davantage après un TLE que G6,
qui corrige plus directement.
Ces diagrammes permettent de confirmer ou d'infirmer cette hypothèse avec la trajectoire complète.

**🇬🇧** We isolate (user, problem) pairs whose **first error is TLE** at level D
and compare the paths followed by groups G4, G5, and G6.
Hypothesis from notebook 06 S4: G4 wanders more after a TLE than G6, which corrects more directly.
These diagrams confirm or refute this hypothesis using the full submission trajectory.

> **Pourquoi TLE ?** Taux de résolution le plus bas tous niveaux confondus (NB06 S2)
> et écart inter-groupes le plus marqué au niveau D (NB06 S4).
>
> *Why TLE? Lowest resolution rate across all levels (NB06 S2) and sharpest group gap at level D (NB06 S4).*

In [ ]:
MIN_PAIRS_GROUP = 15  # seuil plus bas car on découpe par groupe
                      # lower threshold since we split by group

for group in GROUPS:
    total_pairs = (
        df_seq
        .filter(
            (pl.col("difficulty") == "D") &
            (pl.col("first_error") == "TLE") &
            (pl.col("proficiency_group") == group)
        )
        ["n_pairs"].sum()
    )

    fig = build_sankey(
        df_seq,
        difficulty="D",
        first_error_filter="TLE",
        group_filter=group,
        min_pairs=MIN_PAIRS_GROUP,
        title=(
            f"Niveau D, premiere erreur TLE, groupe {group}  ({total_pairs:,} paires)\n"
            f"Level D, first error TLE, group {group}  (sequences with >= {MIN_PAIRS_GROUP} pairs shown)"
        ),
    )

    if fig is not None:
        fig.show()
    else:
        print(f"[{group}] Aucune sequence TLE au niveau D avec n_pairs >= {MIN_PAIRS_GROUP}.")

*Analyse à compléter après exécution / Analysis to be filled after running.*

---
## Section S4 : Répartition des chemins complets depuis TLE × groupe, niveau D
## Complete Path Distribution from TLE × Group, Level D

**🇫🇷** Pour chaque groupe (G4, G5, G6), on liste les chemins complets les plus fréquents
parmi les paires (utilisateur, problème) de niveau D dont la première erreur est TLE.
Les pourcentages sont normalisés sur le total de paires du groupe.
Vert = chemin résolu (→ AC), gris = abandonné.

**🇬🇧** For each group (G4, G5, G6), we list the most frequent complete paths among
level-D (user, problem) pairs whose first error is TLE.
Percentages are normalised over the group's total pairs.
Green = resolved (→ AC), grey = abandoned.

In [ ]:
TOP_N = 12

fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharey=False)

for ax, group in zip(axes, ["G4", "G5", "G6"]):
    df_filt = df_seq.filter(
        (pl.col("difficulty") == "D") &
        (pl.col("first_error") == "TLE") &
        (pl.col("proficiency_group") == group)
    )

    total = df_filt["n_pairs"].sum()

    df_paths = (
        df_filt
        .group_by(["sequence", "resolved"])
        .agg(pl.col("n_pairs").sum())
        .sort("n_pairs", descending=True)
    )

    top      = df_paths.head(TOP_N)
    seqs     = top["sequence"].to_list()
    counts   = top["n_pairs"].to_list()
    resolved = top["resolved"].to_list()
    pcts     = [n / total * 100 for n in counts]
    colors   = ["#2ecc71" if r else "#95a5a6" for r in resolved]

    others_n   = total - top["n_pairs"].sum()
    others_pct = others_n / total * 100

    y_pos = list(range(len(seqs) + 1))
    all_pcts   = pcts + [others_pct]
    all_colors = colors + ["#dfe6e9"]
    all_labels = seqs + ["Autres / Others"]

    bars = ax.barh(y_pos, all_pcts, color=all_colors,
                   edgecolor="white", linewidth=0.5, height=0.72)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(all_labels, fontsize=8, fontfamily="monospace")
    ax.invert_yaxis()
    ax.set_xlabel("% des paires (user, problème) du groupe", fontsize=9)
    ax.set_title(f"{group}  —  N total = {total:,}", fontweight="bold", fontsize=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.spines[["top", "right"]].set_visible(False)

    max_pct = max(all_pcts)
    for bar, pct in zip(bars, all_pcts):
        if pct > 0.3:
            ax.text(
                bar.get_width() + max_pct * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{pct:.1f}%", va="center", ha="left", fontsize=7.5,
            )

legend_elements = [
    mpatches.Patch(facecolor="#2ecc71", label="Résolu (→ AC) / Resolved"),
    mpatches.Patch(facecolor="#95a5a6", label="Abandonné / Abandoned"),
    mpatches.Patch(facecolor="#dfe6e9", edgecolor="#b2bec3",
                   label="Autres chemins / Other paths"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.05))

fig.suptitle(
    "Répartition des chemins complets depuis TLE — Niveau D, par groupe\n"
    "Complete path distribution from TLE — Level D, by proficiency group",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_s4_paths_tle_by_group.png", dpi=150, bbox_inches="tight")
plt.show()

*Analyse à compléter après exécution / Analysis to be filled after running.*

---
## Section 5 — Taux de résolution selon la longueur du chemin
## Resolution Rate by Path Length

Pour un premier TLE au niveau D : est-ce qu'insister (chemin plus long) augmente le taux de résolution ? La longueur du chemin est le nombre de transitions (→) dans la séquence.


In [ ]:
PATH_LEN_MAX = 5  # longueurs >= 5 regroupées dans "5+"

GROUPS   = ["G4", "G5", "G6"]
G_COLORS = {"G4": "#e06c75", "G5": "#e5c07b", "G6": "#98c379"}

fig, ax = plt.subplots(figsize=(10, 6))

for group in GROUPS:
    df_filt = df_seq.filter(
        (pl.col("difficulty") == "D") &
        (pl.col("first_error") == "TLE") &
        (pl.col("proficiency_group") == group)
    )

    # Longueur = nombre de "→" dans la séquence
    df_len = df_filt.with_columns(
        pl.col("sequence").str.count_matches("→").alias("path_len")
    ).with_columns(
        pl.when(pl.col("path_len") >= PATH_LEN_MAX)
          .then(pl.lit(PATH_LEN_MAX))
          .otherwise(pl.col("path_len"))
          .alias("path_len_clamped")
    )

    df_agg = (
        df_len.group_by(["path_len_clamped", "resolved"])
              .agg(pl.col("n_pairs").sum())
              .sort("path_len_clamped")
    )

    lengths, rates, ns = [], [], []
    for l in range(1, PATH_LEN_MAX + 1):
        sub = df_agg.filter(pl.col("path_len_clamped") == l)
        total = sub["n_pairs"].sum()
        if total < 5:
            continue
        res_rows = sub.filter(pl.col("resolved") == True)
        res_n = res_rows["n_pairs"].sum() if res_rows.height > 0 else 0
        lengths.append(str(l) if l < PATH_LEN_MAX else f"{PATH_LEN_MAX}+")
        rates.append(100 * res_n / total)
        ns.append(total)

    ax.plot(lengths, rates, marker='o', color=G_COLORS[group],
            label=f"Groupe {group}", linewidth=2.5, markersize=7)
    # Annoter le N total par longueur (groupe G4 seulement pour lisibilité)
    if group == "G4":
        for x, y, n in zip(lengths, rates, ns):
            ax.annotate(f"n={n}", (x, y), textcoords="offset points",
                        xytext=(0, 8), ha='center', fontsize=7, color='gray')

ax.set_xlabel("Longueur du chemin (nombre de transitions depuis TLE)", fontsize=12)
ax.set_ylabel("Taux de résolution (%)", fontsize=12)
ax.set_title(
    "Taux de résolution selon la longueur du chemin\n"
    "Niveau D — Première erreur TLE",
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()


*Analyse à compléter après exécution / Analysis to be filled after running.*